In [1]:
import pandas as pd
import numpy as np

In [37]:
df = pd.read_csv("gym_churn_synthetic.csv")

In [38]:
df

,Member_ID,Name,Age,Gender,Address,Phone_Number,Membership_Type,Join_Date,Last_Visit_Date,Favorite_Exercise,Avg_Workout_Duration_Min,Avg_Calories_Burned,Total_Weight_Lifted_kg,Visits_Per_Month,Churn
0,GYM100000,Allison Hill,23.0,Female,"42769 Stone Light Apt. 959, New Jacquelinebury...",323.697.0770x540,Monthly,2024-01-21,2026-06-06,Swimming,64.0,469.0,9367,10.3,No
1,GYM100001,Noah Rhodes,39.0,Male,"911 Courtney Streets Suite 975, North Nancy, I...",+1-945-673-5640x8746,Monthly,2025-07-30,2026-03-05,Yoga,72.0,651.0,7137,13.0,Yes
2,GYM100002,Angie Henderson,35.0,Male,"572 Lawrence Mill, Pattonshire, IN 86699",001-976-322-7203x3428,Monthly,2023-08-26,2026-01-12,Swimming,91.0,644.0,4357,8.8,Yes
3,GYM100003,Daniel Wagner,37.0,Female,"312 Thomas Flat Suite 578, North Christopher, ...",001-527-908-2456x3534,Monthly,2025-02-21,2026-06-09,Cardio,66.0,232.0,6224,13.2,No
4,GYM100004,Cristian Santos,32.0,Female,"594 Alex Mall Suite 495, Albertshire, MH 90338",(585)274-8791x81476,Monthly,2025-10-05,2026-05-24,CrossFit,86.0,650.0,21327,17.7,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,GYM149995,Kenneth Cantu,41.0,Female,"7682 Jessica Drive, Hillberg, NE 54561",307.589.7733,Monthly,2024-01-03,2026-06-07,Yoga,92.0,667.0,14073,12.8,No
49996,GYM149996,Carmen Wilson,46.0,Female,"743 Ernest Shoal, South Markview, GU 51561",244.605.5399x073,Monthly,2024-08-11,2026-03-28,Swimming,68.0,421.0,2438,7.9,Yes
49997,GYM149997,Rachel Reed,39.0,Male,"48826 Booth Plains Apt. 198, Port Nicole, AL 7...",2792306951,Yearly,2025-04-26,2026-05-29,Cycling,76.0,511.0,4628,12.9,No
49998,GYM149998,April Edwards,80.0,Female,"6437 John Oval Apt. 510, Josephport, AR 81518",640.412.3941x26860,Monthly,2025-04-05,2026-06-01,Cardio,74.0,615.0,6783,17.8,No


In [39]:
# Engineer New Features for Predicting Churn

from sklearn.preprocessing import LabelEncoder

cat_cols = ['Gender', 'Membership_Type', 'Favorite_Exercise'] # Gender          is   Female=0, Male=1, Other=2
                                                              # Membership_Type is   Monthly=0, Quarterly=1, Yearly=2
                                                              # Favorite_Exercise is Cardio=0, CrossFit=1, Cycling=2 ...
                                                              

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

#Churn Conversion
df['Churn'] = (df['Churn'] == 'Yes').astype(int)


today = pd.Timestamp('2026-06-10')

# Parse raw date strings to datetime 
df['Join_Date']       = pd.to_datetime(df['Join_Date'])
df['Last_Visit_Date'] = pd.to_datetime(df['Last_Visit_Date'])

# Engineer date features 
df['tenure_months']        = (today - df['Join_Date']).dt.days / 30
df['days_since_visit']     = (today - df['Last_Visit_Date']).dt.days

df['is_new_member']        = (df['tenure_months'] < 3).astype(int)
df['is_gym_veteran']           = (df['tenure_months'] > 24).astype(int)
df['visited_this_week']    = (df['days_since_visit'] <= 7).astype(int)

df['visit_rate_vs_tenure'] = df['Visits_Per_Month'] / (df['tenure_months'] + 1)

# Drop the raw date columns 
df = df.drop(columns=['Join_Date', 'Last_Visit_Date'])

In [43]:
df

,Member_ID,Name,Age,Gender,Address,Phone_Number,Membership_Type,Favorite_Exercise,Avg_Workout_Duration_Min,Avg_Calories_Burned,Total_Weight_Lifted_kg,Visits_Per_Month,Churn,tenure_months,days_since_visit,is_new_member,is_gym_veteran,visited_this_week,visit_rate_vs_tenure
0,GYM100000,Allison Hill,23.0,0,"42769 Stone Light Apt. 959, New Jacquelinebury...",323.697.0770x540,0,5,64.0,469.0,9367,10.3,0,29.033333,4,0,1,1,0.342952
1,GYM100001,Noah Rhodes,39.0,1,"911 Courtney Streets Suite 975, North Nancy, I...",+1-945-673-5640x8746,0,7,72.0,651.0,7137,13.0,1,10.500000,97,0,0,0,1.130435
2,GYM100002,Angie Henderson,35.0,1,"572 Lawrence Mill, Pattonshire, IN 86699",001-976-322-7203x3428,0,5,91.0,644.0,4357,8.8,1,33.966667,149,0,1,0,0.251668
3,GYM100003,Daniel Wagner,37.0,0,"312 Thomas Flat Suite 578, North Christopher, ...",001-527-908-2456x3534,0,0,66.0,232.0,6224,13.2,0,15.800000,1,0,0,1,0.785714
4,GYM100004,Cristian Santos,32.0,0,"594 Alex Mall Suite 495, Albertshire, MH 90338",(585)274-8791x81476,0,1,86.0,650.0,21327,17.7,0,8.266667,17,0,0,0,1.910072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,GYM149995,Kenneth Cantu,41.0,0,"7682 Jessica Drive, Hillberg, NE 54561",307.589.7733,0,7,92.0,667.0,14073,12.8,0,29.633333,3,0,1,1,0.417845
49996,GYM149996,Carmen Wilson,46.0,0,"743 Ernest Shoal, South Markview, GU 51561",244.605.5399x073,0,5,68.0,421.0,2438,7.9,1,22.266667,74,0,0,0,0.339542
49997,GYM149997,Rachel Reed,39.0,1,"48826 Booth Plains Apt. 198, Port Nicole, AL 7...",2792306951,2,2,76.0,511.0,4628,12.9,0,13.666667,12,0,0,0,0.879545
49998,GYM149998,April Edwards,80.0,0,"6437 John Oval Apt. 510, Josephport, AR 81518",640.412.3941x26860,0,0,74.0,615.0,6783,17.8,0,14.366667,9,0,0,0,1.158351


In [40]:
# Print the columns of data
df.columns

Index(['Member_ID', 'Name', 'Age', 'Gender', 'Address', 'Phone_Number',
       'Membership_Type', 'Favorite_Exercise', 'Avg_Workout_Duration_Min',
       'Avg_Calories_Burned', 'Total_Weight_Lifted_kg', 'Visits_Per_Month',
       'Churn', 'tenure_months', 'days_since_visit', 'is_new_member',
       'is_gym_veteran', 'visited_this_week', 'visit_rate_vs_tenure'],
      dtype='str')

In [44]:
# Split the Data into feature and target variables
X = df[['Age', 'Gender', 'Membership_Type', 'Favorite_Exercise', 'Avg_Workout_Duration_Min', 
        
        # Time and date features
        'Visits_Per_Month', 'tenure_months', 'days_since_visit', 'is_new_member', 'is_gym_veteran', 'visit_rate_vs_tenure']]

y = df['Churn']


In [45]:
# Train test split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [46]:
# Make train test split
X_train, X_test, y_train, y_test = train_test_split(X,  y,  test_size=0.2, stratify=y, random_state=42)

In [47]:
# Scale the data

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [48]:
# Create Deep Learning model
import torch

In [50]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [51]:
# Create a neural network
import torch.nn as nn

In [52]:
class GymChurnMLP(nn.Module):
    def __init__(self, input_dim):
        super(GymChurnMLP, self).__init__()
        self.network = nn.sequential(

            # First Layer
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Hidden Layer 1
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Hidden Layer 2
            nn.Linear(32, 16),
            nn.ReLU(),
            
            # Output Layer
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.network(x)